# V2a-RSN 220127_F4_run2 — Prospective Graph-State Comparison: Sparse VAR Baseline

Prerequisite: the `sparse_var` connectivity stage must already exist for the selected `RUN_ID`.
Parallel lane: yes, independent across method namespaces.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.

Compare one-state and continuous baselines with K-means and regularized mixtures using inner selection and frozen held-out evaluation. Raw-history purging separates training and future graph targets. Label-permutation tests do not establish neural regimes.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
from effectome.dynamics import GraphStateConfig, TransitionConfig
from effectome.dynamics.state_evaluation import StateEvaluationConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_graph_states_and_transitions,
)

DATASET_ID = "220127_F4_run2"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220127_F4_run2"
RUN_ID = "reference"
METHOD = "sparse_var"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
FORCE = False

GRAPH_CFG = GraphStateConfig(
    n_states=1, feature_representation="signed_lag_offdiag", assignment="kmeans",
    metric="frobenius", pca_components=8, analysis_scope="prospective", seed=42,
)
TRANSITION_CFG = TransitionConfig(laplace=1.0, n_null=200, heldout_fraction=0.3, seed=42)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} graph states and transitions",
    expected_stages=["state_evaluation", "graph_states", "transitions"],
    force=FORCE,
)


In [ ]:
states, transitions = run_graph_states_and_transitions(
    run,
    graph_cfg=GRAPH_CFG,
    transition_cfg=TRANSITION_CFG,
    selection_cfg=StateEvaluationConfig(graph=GRAPH_CFG),
    analysis_scope="prospective",
    on_unavailable="return",
    force=FORCE,
)
if states is None:
    print("State evaluation unavailable:", run.load_completed("state_evaluation").summary())
else:
    print(states.labels.shape, states.metric, states.silhouette)
    print(transitions.transition_matrix)
    print("Conditional label-sequence diagnostic only:", transitions.p_value)
print("Neural regimes remain unvalidated without independent raw-process calibration.")
print_stage_status(run)
